In [ ]:
%pip install pandas db-dtypes google-cloud-bigquery pyarrow scikit-learn

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery

# 1. Configuration
project_id = 'yellow-taxi-trips-2026'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id

# 2. Initialisation du client
client = bigquery.Client(project=project_id)

print(f"✅ Setup réussi ! Connecté au projet : {project_id}")

✅ Setup réussi ! Connecté au projet : yellow-taxi-trips-2026


In [5]:
# Function to execute a BigQuery query and return a DataFrame

def query_to_dataframe(query: str) -> pd.DataFrame:
    """
    Executes a SQL query in BigQuery and returns a Pandas DataFrame.

    Parameters:
    - query (str): The SQL query to execute.

    Return:
    - pd.DataFrame : The DataFrame containing the results of the query.
    """
    try:
        df = client.query(query).to_dataframe()
        print(f"Query executed successfully. Retrieved {df.shape[0]} rows.")
        return df
    except Exception as e:
        print(f"Error executing query: {e}")
        return pd.DataFrame()

In [6]:
query_trips_ml_data = """
SELECT *
FROM `yellow-taxi-trips-2026.ml_dataset.trips_ml_data`
LIMIT 500000
"""
trips_ml_data_df = query_to_dataframe(query_trips_ml_data)
trips_ml_data_df.head()

c:\Users\EDITH ADJE\Documents\GCP Project\NYC_Taxi_GCP\projet_data_taxi_complet\nyc-yellow-taxi-trips\.venvreport\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Query executed successfully. Retrieved 500000 rows.


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file
0,1,2024-11-25 22:20:01+00:00,2024-11-25 22:54:58+00:00,2.0,9.20,5.0,N,70,230,2,0.00,0.0,0.0,0.0,0.0,1.0,1.00,0.0,0.0,yellow_tripdata_2024-11.parquet
1,1,2024-11-20 10:45:57+00:00,2024-11-20 10:53:30+00:00,1.0,1.10,5.0,N,143,238,2,0.01,0.0,0.0,0.0,0.0,1.0,1.01,0.0,0.0,yellow_tripdata_2024-11.parquet
2,2,2024-11-26 22:52:43+00:00,2024-11-27 00:12:25+00:00,1.0,10.29,5.0,N,260,93,2,0.01,0.0,0.0,0.0,0.0,1.0,1.01,0.0,0.0,yellow_tripdata_2024-11.parquet
3,2,2024-11-22 12:30:32+00:00,2024-11-22 12:32:06+00:00,1.0,0.07,5.0,N,1,1,2,0.06,0.0,0.0,0.0,0.0,1.0,1.06,0.0,0.0,yellow_tripdata_2024-11.parquet
4,2,2024-11-16 22:51:19+00:00,2024-11-16 22:52:39+00:00,2.0,0.04,1.0,N,113,113,2,0.00,0.0,0.0,0.0,0.0,0.0,2.50,2.5,0.0,yellow_tripdata_2024-11.parquet


In [7]:
trips_ml_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype              
---  ------                 --------------   -----              
 0   VendorID               500000 non-null  Int64              
 1   tpep_pickup_datetime   500000 non-null  datetime64[us, UTC]
 2   tpep_dropoff_datetime  500000 non-null  datetime64[us, UTC]
 3   passenger_count        500000 non-null  float64            
 4   trip_distance          500000 non-null  float64            
 5   RatecodeID             500000 non-null  float64            
 6   store_and_fwd_flag     500000 non-null  object             
 7   PULocationID           500000 non-null  Int64              
 8   DOLocationID           500000 non-null  Int64              
 9   payment_type           500000 non-null  Int64              
 10  fare_amount            500000 non-null  float64            
 11  extra                  500000 non-null 

In [8]:
# Missing values
trips_ml_data_df.isna().sum()

VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
airport_fee              0
source_file              0
dtype: int64

In [9]:
def preprocess_data(df):
    # Ensure datetime columns are in datetime format
    #df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
    #df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

    # Trip duration in minutes
    df["trip_duration"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

    # Extract time-based features
    df["pickup_dayofweek"] = df["tpep_pickup_datetime"].dt.dayofweek # Monday=0, Sunday=6.
    df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
    df["pickup_year"] = df["tpep_pickup_datetime"].dt.year
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["is_weekend"] = df["pickup_dayofweek"].isin([5, 6]).astype(int)  # 5=Saturday, 6=Sunday

    # Filter payment type (Credit Card = 1, Cash = 2)
    #df = df[df["payment_type"].isin([1, 2])].copy()

    # Create binary feature for credit card payments
    df["is_credit_card"] = (df["payment_type"] == 1).astype(int)

    # Select relevant columns
    selected_cols = [
        "PULocationID", "DOLocationID", "passenger_count", "trip_distance",
        "trip_duration", "pickup_dayofweek", "pickup_month", "pickup_year", "pickup_hour",
        "is_weekend", "is_credit_card", "total_amount"
    ]

    return df[selected_cols].copy()


In [10]:
from sklearn.model_selection import train_test_split

def split_data(df, train_size=0.7, val_size=0.15, test_size=0.15, random_state=42):
    """
    Splits the dataframe into train, validation, and test sets.

    Parameters:
    - df: Pandas DataFrame
    - train_size: Proportion of the dataset for training (default=70%)
    - val_size: Proportion for validation (default=15%)
    - test_size: Proportion for testing (default=15%)
    - random_state: Seed for reproducibility

    Returns:
    - train_df, val_df, test_df: Split DataFrames
    """
    assert train_size + val_size + test_size == 1, "Split sizes must sum to 1"

    # First, split train + val and test
    train_val_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state)

    # Then, split train and validation
    train_df, val_df = train_test_split(train_val_df, test_size=val_size / (train_size + val_size),
                                        random_state=random_state)

    return train_df, val_df, test_df

# Apply the function
train_df, val_df, test_df = split_data(trips_ml_data_df)

# Display the sizes
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")


Train size: 350000
Validation size: 75000
Test size: 75000


In [11]:
preprocessed_train_df = preprocess_data(train_df)
preprocessed_train_df.head()

,PULocationID,DOLocationID,passenger_count,trip_distance,trip_duration,pickup_dayofweek,pickup_month,pickup_year,pickup_hour,is_weekend,is_credit_card,total_amount
413451,144,68,1.0,2.21,12.450000,1,11,2024,12,0,0,18.20
4275,234,170,1.0,0.71,5.733333,4,11,2024,15,0,1,11.28
199488,238,166,1.0,1.30,6.583333,5,11,2024,7,1,0,10.80
389233,237,239,1.0,1.95,18.016667,0,11,2024,17,0,1,24.15
284993,230,246,1.0,1.12,5.216667,2,11,2024,6,0,1,14.28


In [12]:
preprocessed_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 350000 entries, 413451 to 99404
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   PULocationID      350000 non-null  Int64  
 1   DOLocationID      350000 non-null  Int64  
 2   passenger_count   350000 non-null  float64
 3   trip_distance     350000 non-null  float64
 4   trip_duration     350000 non-null  float64
 5   pickup_dayofweek  350000 non-null  int32  
 6   pickup_month      350000 non-null  int32  
 7   pickup_year       350000 non-null  int32  
 8   pickup_hour       350000 non-null  int32  
 9   is_weekend        350000 non-null  int64  
 10  is_credit_card    350000 non-null  int64  
 11  total_amount      350000 non-null  float64
dtypes: Int64(2), float64(4), int32(4), int64(2)
memory usage: 30.0 MB


In [14]:
# Load the preprocessed_train_df dataframe into BigQuery

DATASET_ID = "ml_dataset"
TABLE_ID = "preprocessed_train_data"
FULL_TABLE_ID = f"{project_id}.{DATASET_ID}.{TABLE_ID}"

# Define schema (ensure correct types)
schema = [
    bigquery.SchemaField("PULocationID", "INTEGER"),
    bigquery.SchemaField("DOLocationID", "INTEGER"),
    bigquery.SchemaField("passenger_count", "FLOAT"),
    bigquery.SchemaField("trip_distance", "FLOAT"),
    bigquery.SchemaField("trip_duration", "FLOAT"),
    bigquery.SchemaField("pickup_dayofweek", "INTEGER"),
    bigquery.SchemaField("pickup_month", "INTEGER"),
    bigquery.SchemaField("pickup_year", "INTEGER"),
    bigquery.SchemaField("pickup_hour", "INTEGER"),
    bigquery.SchemaField("is_weekend", "INTEGER"),
    bigquery.SchemaField("is_credit_card", "INTEGER"),
    bigquery.SchemaField("total_amount", "FLOAT"),
]

# Load data into BigQuery
job = client.load_table_from_dataframe(
    preprocessed_train_df, FULL_TABLE_ID, job_config=bigquery.LoadJobConfig(schema=schema)
)

# Wait for the job to complete
job.result()

print(f"Data successfully uploaded to BigQuery: {FULL_TABLE_ID}")

Data successfully uploaded to BigQuery: yellow-taxi-trips-2026.ml_dataset.preprocessed_train_data


In [15]:
preprocessed_test_df = preprocess_data(test_df)
preprocessed_test_df.head()

,PULocationID,DOLocationID,passenger_count,trip_distance,trip_duration,pickup_dayofweek,pickup_month,pickup_year,pickup_hour,is_weekend,is_credit_card,total_amount
104241,170,224,1.0,1.59,7.533333,0,11,2024,10,0,1,14.28
199676,170,107,2.0,0.64,2.333333,6,11,2024,11,1,1,10.92
140199,164,161,1.0,1.25,9.450000,5,11,2024,10,1,1,17.50
132814,237,236,1.0,1.45,7.366667,3,11,2024,16,0,0,13.30
408697,236,140,1.0,2.12,12.583333,6,11,2024,12,1,1,21.84


In [16]:
preprocessed_test_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75000 entries, 104241 to 40007
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PULocationID      75000 non-null  Int64  
 1   DOLocationID      75000 non-null  Int64  
 2   passenger_count   75000 non-null  float64
 3   trip_distance     75000 non-null  float64
 4   trip_duration     75000 non-null  float64
 5   pickup_dayofweek  75000 non-null  int32  
 6   pickup_month      75000 non-null  int32  
 7   pickup_year       75000 non-null  int32  
 8   pickup_hour       75000 non-null  int32  
 9   is_weekend        75000 non-null  int64  
 10  is_credit_card    75000 non-null  int64  
 11  total_amount      75000 non-null  float64
dtypes: Int64(2), float64(4), int32(4), int64(2)
memory usage: 6.4 MB


In [17]:
preprocessed_test_df.shape

(75000, 12)

In [19]:
# Load the preprocessed_test_df dataframe into BigQuery

DATASET_ID = "ml_dataset"
TABLE_ID = "preprocessed_test_data"
FULL_TABLE_ID = f"{project_id}.{DATASET_ID}.{TABLE_ID}"

# Define schema (ensure correct types)
schema = [
    bigquery.SchemaField("PULocationID", "INTEGER"),
    bigquery.SchemaField("DOLocationID", "INTEGER"),
    bigquery.SchemaField("passenger_count", "FLOAT"),
    bigquery.SchemaField("trip_distance", "FLOAT"),
    bigquery.SchemaField("trip_duration", "FLOAT"),
    bigquery.SchemaField("pickup_dayofweek", "INTEGER"),
    bigquery.SchemaField("pickup_month", "INTEGER"),
    bigquery.SchemaField("pickup_year", "INTEGER"),
    bigquery.SchemaField("pickup_hour", "INTEGER"),
    bigquery.SchemaField("is_weekend", "INTEGER"),
    bigquery.SchemaField("is_credit_card", "INTEGER"),
    bigquery.SchemaField("total_amount", "FLOAT"),
]

# Load data into BigQuery
job = client.load_table_from_dataframe(
    preprocessed_test_df, FULL_TABLE_ID, job_config=bigquery.LoadJobConfig(schema=schema)
)

# Wait for the job to complete
job.result()

print(f"Data successfully uploaded to BigQuery: {FULL_TABLE_ID}")

Data successfully uploaded to BigQuery: yellow-taxi-trips-2026.ml_dataset.preprocessed_test_data


In [ ]:
# You can continue to create a custom model
